<a href="https://colab.research.google.com/github/fredgruber/python/blob/main/ingest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import requests
import json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType
from pyspark.sql import SparkSession, functions as F
from pyspark.sql import Row

#ibge_api.ipynb
def fetch_json(url, params=None, timeout=30):
  r = requests.get(url, params=params, timeout=timeout)
  r.raise_for_status()
  return r.json()
  def fetch_populacao_municipal_2024(municipios=None):
  # Exemplo: adaptar ao endpoint real de IBGE para população 2024
    url = "https://servicodados.ibge.gov.br/api/v3/agregados/6579/"
    if municipios:
        url += f"?municipios={','.join(map(str, municipios))}"
    return fetch_json(url)
    def fetch_pib_municipal_2021(municipios=None):
      # Exemplo: adaptar ao endpoint real de IBGE para PIB 2021
      url = "https://servicodados.ibge.gov.br/api/v3/agregados/5938/"
      if municipios:
          url += f"?municipios={','.join(map(str, municipios))}"
  return fetch_json(url)

#bronze
from google.colab import drive
drive.mount('/content/drive')
def read_json_to_df(spark, json_data, schema):
  return spark.read.json(spark.sparkContext.parallelize([json.dumps(json_data)]), schema=schema)
def main():
  spark = SparkSession.builder.appName("IBGE-Ingest").getOrCreate()

  # Exemplo de dados simulados - substituir com mapeamento real vindo da API
  populacao_2024 = fetch_populacao_municipal_2024()  # retornar lista de dicts
  pib_2021 = fetch_pib_municipal_2021()

  # Defina schemas reais conforme o retorno da API IBGE
  pop_schema = StructType([
      StructField("codigo_ibge", StringType(), True),
      StructField("municipio", StringType(), True),
      StructField("estado", StringType(), True),
      StructField("ano", IntegerType(), True),
      StructField("populacao", LongType(), True),
      StructField("fonte", StringType(), True),
  ])
  pib_schema = StructType([
      StructField("codigo_ibge", StringType(), True),
      StructField("municipio", StringType(), True),
      StructField("estado", StringType(), True),
      StructField("ano", IntegerType(), True),
      StructField("pib", DoubleType(), True),
      StructField("fonte", StringType(), True),
  ])

  # Converter listas de dicts para DataFrame Spark
  df_pop = spark.createDataFrame([Row(**d) for d in populacao_2024], schema=pop_schema)
  df_pib = spark.createDataFrame([Row(**d) for d in pib_2021], schema=pib_schema)

  # Bronze -> salvar Parquet
  df_pop.write.mode("overwrite").parquet("/content/drive/Colab_Notebooks/Exercicio_Amorzao/data/raw/populacao_2024.parquet")
  df_pib.write.mode("overwrite").parquet("/content/drive/Colab_Notebooks/Exercicio_Amorzao/data/raw/pib_municipal_2021.parquet")

  # Silver (limpeza/normalização)
  df_pop_silver = df_pop.select(
      F.col("codigo_ibge").alias("codigo_ibge"),
      F.col("municipio").alias("municipio"),
      F.col("estado").alias("estado"),
      F.col("ano"),
      F.col("populacao").alias("populacao_estimativa"),
      F.lit("IBGE-População-2024").alias("fonte")
  )
  df_pib_silver = df_pib.select(
      F.col("codigo_ibge").alias("codigo_ibge"),
      F.col("municipio").alias("municipio"),
      F.col("estado").alias("estado"),
      F.col("ano"),
      F.col("pib").alias("pib_municipal"),
      F.lit("IBGE-PIB-2021").alias("fonte")
  )

  df_pop_silver.write.mode("overwrite").parquet("/content/drive/Colab_Notebooks/Exercicio_Amorzao/data/silver/populacao_2024.parquet")
  df_pib_silver.write.mode("overwrite").parquet("/content/drive/Colab_Notebooks/Exercicio_Amorzao/data/silver/pib_municipal_2021.parquet")

  spark.stop()

from pyspark.sql import functions as F


def compute_indicators(spark):
  # Carregar silver/parquet já existentes
  df_pop = spark.read.parquet("/content/drive/Colab_Notebooks/Exercicio_Amorzao/data/silver/populacao_2024.parquet")
  df_pib = spark.read.parquet("/content/drive/Colab_Notebooks/Exercicio_Amorzao/data/silver/pib_municipal_2021.parquet")

  # Exemplo: join por municipio para calcular PIB per capita
  df = df_pib.join(df_pop, ["codigo_ibge", "municipio", "estado", "ano"]).withColumn("pib_per_capita", F.col("pib_municipal") / F.col("populacao_estimativa"))

  df.write.mode("overwrite").parquet("/content/drive/Colab_Notebooks/Exercicio_Amorzao/data/gold/indicators.parquet")
  return df

!pip install dash
!pip install dash-bootstrap-components
import dash
from dash import dash_table, dcc, html
import dash_bootstrap_components as dbc
import pandas as pd


app = dash.Dash("DashIBGE", external_stylesheets=[dbc.themes.BOOTSTRAP])

import pyarrow.parquet as pq
df = pq.read_table("/content/drive/Colab_Notebooks/Exercicio_Amorzao/data/gold/indicators.parquet").to_pandas()


top_pib = df.sort_values("pib_municipal", ascending=False).head(10)


app.layout = html.Div([
html.H1("IBGE - Indicadores Municipais (2021/2024)"),
dash_table.DataTable(
    id="tbl-top-pib",
    columns=[{"name": c, "id": c} for c in top_pib.columns],
    data=top_pib.to_dict("records"),
    page_size=10,
),
dcc.Graph(
    id="bar-pib",
    figure={
        "data": [
            {"x": top_pib["municipio"][:10], "y": top_pib["pib_municipal"][:10], "type": "bar", "name": "PIB"}
        ],
        "layout": {"title": "PIB Municipal (top 10) - 2021"}
    }
),
])


if name == "main":
  app.run_server(debug=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: /content/drive/Colab_Notebooks/Exercicio_Amorzao/data/gold/indicators.parquet